In [ ]:
import sys
import os

# Add the parent directory (src) to the Python path
sys.path.append(os.path.dirname(os.getcwd()))

In [0]:
from utility.catalog_utils import CatalogUtils
from utility.dataquality_utils import DataQuality
import yaml

In [0]:
current_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

# Get the directory containing the notebook (remove the notebook filename)
src_directory = '/'.join(current_notebook_path.split('/')[:-1])

# Build the CSV file path
file_path = f"/Workspace{src_directory}/util/parameters.yml"
print(f"CSV file path: {file_path}")

with open(file_path, "r") as file:
    config = yaml.safe_load(file)

In [0]:
# Data Quality check table variable
data_quality_catalog = config["data_quality_catalog"]
data_quality_schema = config["data_quality_schema"]
data_quality_table = config["data_quality_table"]

# config table variable
config_catalog = config["config_catalog"]
config_schema = config["config_schema"]
config_table = config["config_table"]

In [0]:
layer_name_dq= task_name = dbutils.widgets.get("layer_name")

In [0]:
CatalogUtils.ensure_catalog_and_schema(spark, data_quality_catalog, data_quality_schema)

In [0]:
# Initialze the class
DQ = DataQuality(spark,data_quality_catalog,data_quality_schema,data_quality_table)

In [0]:
distinct_catalog_schema= spark.sql(f"select * from {config_catalog}.{config_schema}.{config_table}").select('target_table_catalog','target_table_schema').distinct().collect()

In [0]:
for row_ in distinct_catalog_schema:
    print(f"Data Quality Check Start for Catalog: {row_.target_table_catalog}.{row_.target_table_schema} --> ")
    bronze_result = DQ.dq_check(row_.target_table_catalog, row_.target_table_schema,layer_name_dq)
    print(f"Data Quality Check done for Catalog: {row_.target_table_catalog}.{row_.target_table_schema}.")

Row(target_table_catalog='poc_catalog', target_table_schema='pg_sql_migration')
poc_catalog
pg_sql_migration
